In [8]:
import pandas as pd

df = pd.read_parquet("data/clustered_data_gen2_depth4.parquet")
cluster1 = df[df["cluster"] == 1].copy()
cluster1["year"] = pd.to_datetime(cluster1["date"]).dt.year

print("Full year breakdown for cluster 1:")
print(cluster1["year"].value_counts().sort_index())

Full year breakdown for cluster 1:
year
2016     3376
2017    10634
2018     6993
2019     6382
2020     4044
2021     4283
Name: count, dtype: int64


In [9]:
import numpy as np
import pandas as pd

from clusters import PRIM_EXCLUDE, SEC_EXCLUDE, ID_COLS, TARGET_COL

df1 = pd.read_parquet("data/clustered_data_gen2_depth4.parquet")

exclude = set(ID_COLS) | {TARGET_COL, "cluster", "year"} | PRIM_EXCLUDE | SEC_EXCLUDE
feature_cols = [c for c in df1.select_dtypes(include=[np.number]).columns if c not in exclude]

print(f"Comparing {len(feature_cols)} features (same set used for clustering)\n")

cluster_means = df1.groupby("cluster")[feature_cols].mean().T
overall_std = df1[feature_cols].std()

cluster_means["diff_1_vs_0"] = cluster_means[1] - cluster_means[0]
cluster_means["diff_in_stds"] = cluster_means["diff_1_vs_0"] / overall_std

result = cluster_means[[0, 1, "diff_1_vs_0", "diff_in_stds"]].sort_values(
    "diff_in_stds", key=abs, ascending=False
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
print(result.head(30).to_string())

Comparing 69 features (same set used for clustering)

cluster                                       0          1  diff_1_vs_0  diff_in_stds
1y_trend                               0.002893   0.001257    -0.001636     -1.547388
5y_trend                               0.001236   0.000640    -0.000596     -1.444134
ret_5y                                 6.230798   1.759100    -4.471699     -1.398780
ret_1y                                 1.157502   0.410312    -0.747190     -1.354442
1y_drawdown                           -0.058897  -0.023631     0.035267      1.307673
5y_drawdown                           -0.065632  -0.027249     0.038383      1.220978
ret_6m                                 0.547383   0.245875    -0.301508     -1.142044
ret_3y                                 3.057952   0.828670    -2.229282     -0.972173
sec_positive_5y_trend_pct_market       0.706512   0.722309     0.015797      0.454395
sec_avg_5y_trend_market                0.000158   0.000176     0.000018      0.376326


In [10]:
import numpy as np
import pandas as pd

from clusters import PRIM_EXCLUDE, SEC_EXCLUDE, ID_COLS, TARGET_COL

df1 = pd.read_parquet("data/clustered_data_gen2_depth4.parquet")

exclude = set(ID_COLS) | {TARGET_COL, "cluster", "year"} | PRIM_EXCLUDE | SEC_EXCLUDE
feature_cols = [c for c in df1.select_dtypes(include=[np.number]).columns if c not in exclude]

print(f"Comparing {len(feature_cols)} features (same set used for clustering)\n")

cluster_means = df1.groupby("cluster")[feature_cols].mean().T
overall_std = df1[feature_cols].std()

cluster_means["diff_1_vs_0"] = cluster_means[1] - cluster_means[0]
cluster_means["diff_in_stds"] = cluster_means["diff_1_vs_0"] / overall_std

result = cluster_means[[0, 1, "diff_1_vs_0", "diff_in_stds"]].sort_values(
    "diff_in_stds", key=abs, ascending=False
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
print(result.head(60).tail(30).to_string())

Comparing 69 features (same set used for clustering)

cluster                                       0         1  diff_1_vs_0  diff_in_stds
beta_1y                                2.104688  1.966421    -0.138267     -0.224124
ret_1w                                 0.018317  0.011625    -0.006692     -0.201757
sector_Healthcare                      0.160046  0.103915    -0.056131     -0.177879
sec_high_monotonic_pct                 0.298392  0.317399     0.019006      0.164307
sector_Real Estate                     0.004884  0.031166     0.026282      0.161252
monotonic_score                        0.815912  0.800063    -0.015850     -0.147227
sec_positive_1y_trend_pct_market       0.694968  0.673699    -0.021269     -0.142418
sector_Communication Services          0.056822  0.031054    -0.025768     -0.140528
sector_Basic Materials                 0.019700  0.046455     0.026755      0.132583
sec_avg_monotonic_score                0.630432  0.638318     0.007885      0.122182
sec_avg_ret